# Would this method have found an effect that was not there?

An estimator's 95% interval is a promise: run the experiment on a world with no effect, and it
will exclude zero one time in twenty. Nobody checks. The method gets picked because it is
standard, and if its intervals are actually 88% intervals, that shows up as a series of
findings that fail to replicate — years later, attributed to anything else.

An A/A calibration is the check, and it is cheap: simulate the null world, run the method, count.
This notebook does it for every method in the registry, and judges the count against an exact
binomial acceptance region rather than against a hopeful threshold.

One `SimulationSpec` describes one panel and every method reads the same draw of it. A/A
calibration forces `effect = 0` and counts the runs whose interval at mass `1 − α` excludes
zero; the count is judged against the exact binomial region `clopper_pearson(n, α, 1e-3)`,
and an `Unsupported` return is counted separately (more than 10 % of them fails the method
too). A/B power scores rejections, bias, RMSE and coverage at a non-zero effect.

Under this DGP the DiD estimator's standard error is exact:
`se = noise_sd · sqrt(1/n_pre + 1/n_post) · sqrt(1/n_treated + 1/n_control)` — the unit
intercepts cancel in each unit's change and the common shocks cancel in the contrast.

In [ ]:
import numpy as np

from axiom.design import (
    CalibrationResult, Leaderboard, LeaderboardRow, METHODS, PanelDesign, SimulatedPower,
    SimulationSpec, calibrate_method, calibrate_registry, design_for_method,
    difference_in_differences_se, leaderboard, power_from_se, simulate_panel, simulated_power,
)

from axiom.display import enable, table

import sys; sys.path[:0] = ["..", "../.."]  # nbs/ is on the path either way
from _style import CRITICAL, GOOD, caption, compare, intervals, mark_x, shade

enable();  # every axiom result renders itself from here on

In [ ]:
spec = SimulationSpec(n_units=16, n_periods=12, n_pre=6, n_treated=8, noise_sd=1.0, n_simulations=100, seed=11)
print("n_post:", spec.n_post, "n_control:", spec.n_control, "alpha:", spec.alpha)
rows = []
for m in METHODS:
    d: PanelDesign = design_for_method(m)
    rows.append([m, str(d)])
table(rows, headers=("method", "panel design it needs"))
panel = simulate_panel(spec, np.random.default_rng(0), design="holdout")
print("one panel:", panel.outcome.shape)

In [ ]:
cal: CalibrationResult = calibrate_method("difference_in_differences", spec, alpha=0.05)
print(f"false positives {cal.false_positive_count}/{cal.n_evaluated} = {cal.false_positive_rate:.3f}")
print(f"acceptance region [{cal.region.lower}, {cal.region.upper}] at nominal {cal.alpha}; passed={cal.passed} {cal.reason}")

In [ ]:
se = difference_in_differences_se(spec)
predicted = power_from_se(0.8, se).power
ab = spec.model_copy(update={"effect": 0.8})
sp: SimulatedPower = simulated_power("difference_in_differences", ab, predicted_power=predicted)
print(f"exact DiD se {se:.4f}; predicted power {predicted:.3f}; realized {sp.power:.3f} ({sp.rejections}/{sp.n_evaluated})")
print(f"bias {sp.bias:+.3f}  rmse {sp.rmse:.3f}  coverage {sp.coverage:.2f}  within prediction: {sp.within_prediction}")

## Calibrating the registry

`calibrate_registry` returns a *new* read-only registry with each method's status set from its
A/A result; `METHODS` is never mutated. `leaderboard` ranks methods by mean power across effect
sizes at their calibrated size, calibrated methods first.

In [ ]:
small = spec.model_copy(update={"n_simulations": 40})
registry, results = calibrate_registry(small, alpha=0.05)
table(
    [
        [r.method, r.design, f"{r.false_positive_count}/{r.n_evaluated}", str(r.passed),
         registry[r.method].status]
        for r in results
    ],
    headers=("method", "design", "false positives", "passed", "status"),
)

In [ ]:
rows = [
    (f"{r.method}  ({'passed' if r.passed else 'FAILED'})", r.false_positive_rate,
     r.region.lower / r.n_evaluated, r.region.upper / r.n_evaluated)
    for r in results
]
failing = [label for label, *_ in rows if "FAILED" in label]
fig = intervals(
    rows, ref=0.05, ref_label="nominal 5%",
    highlight=failing[0] if failing else None,
    title="Does the interval keep its promise?",
    subtitle="observed false-positive rate per method, with the exact binomial acceptance region it is judged against",
    x_title="rate at which a null world produced a 'finding'",
)
caption(fig, "The bar through each point is not an error bar — it is the region of counts "
             "that an honest 5% method can produce at this number of simulations. A method "
             "whose point sits outside its own bar is not noisy, it is miscalibrated, and its "
             "status in the returned registry says so.")

In [ ]:
board: Leaderboard = leaderboard(small, effects=(0.8, 1.6), alpha=0.05)
rows = []
for row in board.rows:
    assert isinstance(row, LeaderboardRow)
    rows.append(
        [row.rank, row.method, str(row.calibrated),
         str(tuple(round(p, 2) for p in row.powers)), f"{row.mean_power:.2f}"]
    )
table(rows, headers=("rank", "method", "calibrated", "powers", "mean power"))
print(board.row("difference_in_differences").coverages)

In [ ]:
ranked = {f"{row.method}{'' if row.calibrated else '  (uncalibrated)'}": row.mean_power for row in board.rows}
fig = compare(
    list(ranked), list(ranked.values()),
    highlight=list(ranked)[0],
    value_fmt="{:.2f}",
    title="…and only then, which one finds things",
    subtitle="mean power across effects of 0.8 and 1.6, at each method's own calibrated size",
    x_title="mean power",
)
caption(fig, "Power is the second question. Ranking on it before the calibration check "
             "rewards exactly the methods whose intervals are too narrow — they reject more "
             "often at every effect size, including zero.")

## What this bought you

Every estimator in the registry checked against a null world before it is allowed to rank on
power, with the pass/fail judged by an exact binomial region at the stated N — and a
calibrated registry returned as a *new* object, so the check cannot be silently overwritten by
the last run.